# Exploratory Data Analysis

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# Connect to in-memory DuckDB engine
con = duckdb.connect()

BASE_DIR = Path.cwd().resolve().parent.parent
DATA_DIR = BASE_DIR / "data" / "sample"

print(BASE_DIR)
print(DATA_DIR)

## Inspection of the three tables

### Call center interactions table

In [ ]:
# Inspect call_center_interactions schema
con.sql(f"DESCRIBE SELECT * FROM '{DATA_DIR}/call_center_interactions/*.csv' LIMIT 1").show()

In [ ]:
interactions_df = con.sql(f"SELECT * FROM '{DATA_DIR}/call_center_interactions/*.csv' ").to_df()

In [ ]:
interactions_df.info()

In [ ]:
interactions_df.interaction_type.value_counts()

### Call transcripts table

In [ ]:
# Inspect call_transcripts schema
con.sql(f"DESCRIBE SELECT * FROM '{DATA_DIR}/call_transcripts/*.csv' LIMIT 1").show()

In [ ]:
transcripts_df = con.sql(f"SELECT * FROM '{DATA_DIR}/call_transcripts/*.csv' ").to_df()
transcripts_df.info()

In [ ]:
transcripts_df.interaction_id.value_counts()

In [ ]:
### Complaints table

In [ ]:
# Inspect complaints schema
con.sql(f"DESCRIBE SELECT * FROM '{DATA_DIR}/complaints/*.csv' LIMIT 1").show()

In [ ]:
complaints_df = con.sql(f"SELECT * FROM '{DATA_DIR}/complaints/*.csv' ").to_df()

In [ ]:
complaints_df.info()

## Column Value Distribution Analysis

In [ ]:
# Check contact reasons and escalation flags
con.sql(f"""
    SELECT reason_category, was_escalated, COUNT(*) as count
    FROM '{DATA_DIR}/call_center_interactions/*.csv'
    GROUP BY 1, 2
    ORDER BY count DESC
    LIMIT 20
""").show()

In [ ]:
# Check complaint categories and case types
con.sql(f"""
    SELECT case_type, category, COUNT(*) as count
    FROM '{DATA_DIR}/complaints/*.csv'
    GROUP BY 1, 2
    ORDER BY count DESC
    LIMIT 20
""").show()

## Join Verification & Class Balance Check

In [ ]:
df = con.sql(f"""
    SELECT 
        t.customer_text AS text,
        i.reason_category,
        i.was_escalated,
        c.category AS complaint_category,
        c.case_type AS complaint_case_type
    FROM '{DATA_DIR}/call_transcripts/*.csv' t
    JOIN '{DATA_DIR}/call_center_interactions/*.csv' i
        ON t.interaction_id = i.interaction_id
    LEFT JOIN '{DATA_DIR}/complaints/*.csv' c
        ON i.interaction_id = c.origin_interaction_id
    WHERE t.customer_text IS NOT NULL AND LENGTH(TRIM(t.customer_text)) > 10
""").df()

# Display summary statistics
print(f"Total merged rows: {len(df)}")
df.head()

In [ ]:
df["complaint_category"].isnull().value_counts()

In [ ]:
df["complaint_case_type"].isnull().value_counts()